# ice9 free tier

This notebook walks through the free tier results. Run the first cell once to submit your image, then explore each service's output in the cells below.

**Before you start:**
- Set your API key: `export ICE9_API_KEY=ice9_...` in your terminal before launching Jupyter, or set it in the cell below.
- Install dependencies: `pip install ice9`

In [ ]:
from ice9 import Ice9
from ice9.exceptions import PartialResultError

# Set your image path here
IMAGE = "path/to/your/image.jpg"

# If you didn't set ICE9_API_KEY in your environment, you can set it here instead:
# import os
# os.environ["ICE9_API_KEY"] = "ice9_..."

client = Ice9()

try:
    result = client.analyze(IMAGE, tier="free")
except PartialResultError as e:
    print(f"Warning: some services failed: {e.result.services_failed}")
    result = e.result

print(f"Done. Image ID: {result.image_id}")
print(f"Services: {result.services_submitted}")

## Content moderation — nudenet

nudenet detects explicit content. Each detection has a `label` (what was found) and a `confidence` (how sure the model is, from 0 to 1).

In [ ]:
if result.nudenet is not None:
    if result.nudenet.predictions:
        for detection in result.nudenet.predictions:
            print(f"{detection['label']}  confidence={detection['confidence']:.0%}")
    else:
        print("No detections.")

You can filter to just the labels that most applications care about using `CENSOR_LABELS`:

In [ ]:
from ice9 import CENSOR_LABELS

for detection in result.nudenet.predictions:
    if detection["label"] in CENSOR_LABELS and detection["confidence"] >= 0.5:
        print(f"{detection['label']}  confidence={detection['confidence']:.0%}")

## Content analysis

Content analysis combines the nudenet detections into a higher-level assessment of the image — scene type, what anatomy is present, gender breakdown, and intimacy level. This is the synthesised conclusion; nudenet above is the raw evidence it's based on.

In [ ]:
if result.content_analysis is not None:
    full_analysis = result.content_analysis.full_analysis
    activity_analysis = full_analysis.get("activity_analysis", {})
    print(f"Scene type:      {activity_analysis.get('scene_type', '—')}")
    print(f"Intimacy level:  {activity_analysis.get('intimacy_level', '—')}")
    print(f"People count:    {activity_analysis.get('people_count', 0)}")
    print(f"Anatomy exposed: {full_analysis.get('anatomy_exposed', [])}")
else:
    print("No content analysis.")

In [ ]:
# Gender breakdown — confidence and vote details
if result.content_analysis is not None:
    full_analysis = result.content_analysis.full_analysis
    print(full_analysis.get("gender_breakdown"))

In [ ]:
# Full analysis — all the detail if you need it
import json
if result.content_analysis is not None:
    print(json.dumps(result.content_analysis.full_analysis, indent=2, default=str))

## Colors

The dominant colors in the image, as hex codes.

In [ ]:
if result.colors is not None:
    print(result.colors.dominant)

## Metadata

File format, dimensions, and EXIF data.

In [ ]:
if result.metadata is not None:
    for key, value in result.metadata._data.items():
        print(f"{key}: {value}")

## OCR — text in the image

In [ ]:
if result.ocr is not None:
    text = result.ocr._data.get("text") or ""
    if text.strip():
        print(text)
    else:
        print("No text found.")

## QR codes and barcodes

In [ ]:
if result.qr is not None:
    codes = result.qr._data.get("codes") or []
    if codes:
        for code in codes:
            print(f"[{code.get('type', '?')}] {code.get('data', '')}")
    else:
        print("No codes found.")

## Censoring the image

If nudenet found anything, you can draw over the flagged regions using `result.censor()`.

Requires Pillow: `pip install Pillow`

In [ ]:
from IPython.display import display

censored = result.censor(IMAGE, method="pixelate")
display(censored)

## Full result as JSON

Everything in one dict, useful for saving to a file or sending to another service.

In [ ]:
print(result.to_json(indent=2))